# Databricks-AzureML Integration Test

Unified notebook that tests the integration between Databricks and Azure ML.

## Prerequisites
- Databricks secret scope: `azureml-kv-scope` with the following secrets:
  - `subscription-id`: Azure subscription ID
  - `resource-group`: Azure resource group name
  - `workspace-name`: Azure ML workspace name
  - `databricks-host`: Databricks workspace URL
  - `databricks-token`: Databricks API token
  - `mlflow-model-name`: MLflow model name
  - `azureml-endpoint-name`: Azure ML endpoint name

## Tests Performed
1. ✓ Databricks → Azure ML (call endpoint)
2. ✓ Azure ML → Databricks (load MLflow model)
3. ✓ Databricks Workspace Client API
4. ✓ Integration Summary Report

## 1. Setup and Configuration

In [ ]:
import os
import json
import logging
from datetime import datetime

# Azure and Databricks SDKs
from azure.identity import DefaultAzureCredential, ClientSecretCredential, ChainedTokenCredential, EnvironmentCredential
from databricks.sdk import WorkspaceClient
import mlflow
import mlflow.pyfunc
import requests

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✓ All required packages imported successfully")

In [ ]:
# Retrieve secrets from Databricks secret scope
secret_scope = "azureml-kv-scope"

try:
    subscription_id = dbutils.secrets.get(scope=secret_scope, key="subscription-id")
    resource_group = dbutils.secrets.get(scope=secret_scope, key="resource-group")
    workspace_name = dbutils.secrets.get(scope=secret_scope, key="workspace-name")
    databricks_host = dbutils.secrets.get(scope=secret_scope, key="databricks-host")
    databricks_token = dbutils.secrets.get(scope=secret_scope, key="databricks-token")
    mlflow_model_name = dbutils.secrets.get(scope=secret_scope, key="mlflow-model-name")
    azureml_endpoint_name = dbutils.secrets.get(scope=secret_scope, key="azureml-endpoint-name")
    
    print("✓ Retrieved Azure ML credentials from Databricks secret scope")
    print(f"  Subscription ID: {subscription_id[:20]}...")
    print(f"  Resource Group: {resource_group}")
    print(f"  AzureML Workspace: {workspace_name}")
    print(f"  Databricks Host: {databricks_host[:40]}...")
    print(f"  MLflow Model: {mlflow_model_name}")
    print(f"  Azure ML Endpoint: {azureml_endpoint_name}")
except Exception as e:
    print(f"✗ Error retrieving secrets: {e}")
    print(f"\nCreate secret scope with: dbutils.secrets.createScope('{secret_scope}')")
    raise

## 2. Test 1: Databricks → Azure ML (Call Endpoint)

In [ ]:
test_1_passed = False
test_1_result = {"status": "failed", "message": ""}

try:
    print("\n=== Test 1: Databricks → Azure ML ===")
    print(f"Calling Azure ML endpoint: {azureml_endpoint_name}")
    
    # Authenticate to Azure
    tenant_id = os.getenv("AZURE_TENANT_ID")
    client_id = os.getenv("AZURE_CLIENT_ID")
    client_secret = os.getenv("AZURE_CLIENT_SECRET")
    
    # Use ChainedTokenCredential for robust authentication
    if all([tenant_id, client_id, client_secret]):
        credential = ChainedTokenCredential(
            EnvironmentCredential(),
            ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret),
            DefaultAzureCredential()
        )
        print("Using service principal credentials")
    else:
        credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
        print("Using default credentials (managed identity)")
    
    # Get token for Azure ML
    token = credential.get_token("https://ml.azure.com/.default").token
    print(f"✓ Authenticated to Azure ML")
    
    # Construct endpoint URL
    azureml_endpoint_url = f"https://{resource_group}.{workspace_name}.inference.ml.azure.com/score"
    
    # Sample payload (adjust based on your model)
    payload = {
        "data": [[1.0, 2.0, 3.0, 4.0]]
    }
    
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    response = requests.post(azureml_endpoint_url, json=payload, headers=headers, timeout=30)
    
    if response.status_code == 200:
        print(f"✓ Azure ML Endpoint Response: {response.status_code}")
        print(f"  Result: {response.json()}")
        test_1_passed = True
        test_1_result = {"status": "passed", "message": "Successfully called Azure ML endpoint"}
    else:
        print(f"✗ Endpoint returned status {response.status_code}")
        print(f"  Response: {response.text}")
        test_1_result = {"status": "failed", "message": f"Status code: {response.status_code}"}
        
except Exception as e:
    print(f"✗ Error: {e}")
    test_1_result = {"status": "failed", "message": str(e)}

## 3. Test 2: Azure ML → Databricks (Load MLflow Model)

In [ ]:
test_2_passed = False
test_2_result = {"status": "failed", "message": ""}

try:
    print("\n=== Test 2: Azure ML → Databricks (Load MLflow Model) ===")
    print(f"Loading model: {mlflow_model_name}")
    
    # Set up Databricks as MLflow remote
    mlflow.set_tracking_uri("databricks")
    mlflow.set_registry_uri("databricks://workspace")
    
    print(f"✓ MLflow configured for Databricks")
    print(f"  Tracking URI: {mlflow.get_tracking_uri()}")
    print(f"  Registry URI: {mlflow.get_registry_uri()}")
    
    # Load model from Databricks MLflow registry
    model_uri = f"models:/{mlflow_model_name}/latest"
    model = mlflow.pyfunc.load_model(model_uri)
    
    print(f"✓ Successfully loaded MLflow model: {mlflow_model_name}")
    print(f"  Model type: {type(model)}")
    
    test_2_passed = True
    test_2_result = {"status": "passed", "message": f"Successfully loaded model {mlflow_model_name}"}
    
except Exception as e:
    print(f"✗ Error loading MLflow model: {e}")
    test_2_result = {"status": "failed", "message": str(e)}

## 4. Test 3: Databricks Workspace Client API

In [ ]:
test_3_passed = False
test_3_result = {"status": "failed", "message": ""}

try:
    print("\n=== Test 3: Databricks Workspace Client API ===")
    
    # Initialize Databricks client
    ws = WorkspaceClient(host=databricks_host, token=databricks_token)
    
    print(f"✓ Connected to Databricks workspace")
    print(f"  Host: {databricks_host}")
    
    # List workspace contents
    workspace_items = list(ws.workspace.list_workspace("/Users"))
    
    print(f"\n✓ Listed workspace contents ({len(workspace_items)} items)")
    print(f"  Sample items:")
    for item in workspace_items[:5]:
        if hasattr(item, 'path'):
            print(f"    - {item.path}")
    
    test_3_passed = True
    test_3_result = {"status": "passed", "message": f"Successfully listed {len(workspace_items)} workspace items"}
    
except Exception as e:
    print(f"✗ Error: {e}")
    test_3_result = {"status": "failed", "message": str(e)}

## 5. Integration Test Summary Report

In [ ]:
# Generate integration test summary

summary = {
    "timestamp": datetime.now().isoformat(),
    "test_results": {
        "test_1_databricks_to_azureml": test_1_result,
        "test_2_azureml_to_databricks_mlflow": test_2_result,
        "test_3_databricks_workspace_api": test_3_result
    },
    "overall_status": "passed" if all([test_1_passed, test_2_passed, test_3_passed]) else "failed",
    "environment": {
        "subscription_id": subscription_id[:8] + "..." if subscription_id else "N/A",
        "resource_group": resource_group,
        "workspace_name": workspace_name,
        "mlflow_model": mlflow_model_name,
        "azureml_endpoint": azureml_endpoint_name
    }
}

print("\n" + "="*70)
print("DATABRICKS-AZUREML INTEGRATION TEST SUMMARY")
print("="*70)
print(json.dumps(summary, indent=2))
print("="*70)

# Overall status
if summary["overall_status"] == "passed":
    print("\n✓ ALL TESTS PASSED!")
    print("✓ Your Databricks-AzureML integration is working correctly.")
else:
    print("\n✗ SOME TESTS FAILED")
    print("Check the test results above for details and troubleshooting steps.")